<a href="https://colab.research.google.com/github/Touqeerahmed7/Bootstrap/blob/main/Advance_AI_WisdomNet_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

X, y = make_moons(n_samples=1200, noise=0.2, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

class BaseNet(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=16, num_classes=2):
        super(BaseNet, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.out_layer = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        features = self.feature_extractor(x)
        out = self.out_layer(features)
        return out

base_net = BaseNet(input_dim=2, hidden_dim=16, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(base_net.parameters(), lr=0.01)

print("Pre-training Base Network")
for epoch in range(200):
    optimizer.zero_grad()
    outputs = base_net(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

base_net.eval()
with torch.no_grad():
    test_preds = torch.argmax(base_net(X_test_t), dim=1)
    acc = (test_preds == y_test_t).float().mean().item()
    err = 1.0 - acc
print(f"BaseNet Test Accuracy: {acc * 100:.2f}%, Test Error: {err * 100:.2f}%\n")

base_net.eval()
with torch.no_grad():
    train_preds = torch.argmax(base_net(X_train_t), dim=1)
    misclassified_mask = train_preds != y_train_t

S_r_x = X_train_t[misclassified_mask]

REJECT_LABEL = 2
S_r_y = torch.full((S_r_x.size(0),), REJECT_LABEL, dtype=torch.long)

print(f"Identified {len(S_r_x)} misclassified training samples for fine-training (Reject Set Sr).")

class WisdomNet(nn.Module):
    def __init__(self, base_model):
        super(WisdomNet, self).__init__()
        self.feature_extractor = base_model.feature_extractor

        hidden_dim = base_model.out_layer.in_features
        self.out_layer = nn.Linear(hidden_dim, 3)

        with torch.no_grad():
            self.out_layer.weight[:2, :] = base_model.out_layer.weight.detach()
            self.out_layer.bias[:2] = base_model.out_layer.bias.detach()

            self.out_layer.weight[2, :] = 0.0
            self.out_layer.bias[2] = 0.0

    def forward(self, x):
        features = self.feature_extractor(x)
        out = self.out_layer(features)
        return out

wisdom_net = WisdomNet(base_net)

print("\n Fine-training WisdomNet on Reject Set:")
fine_optimizer = optim.Adam(wisdom_net.parameters(), lr=0.005)

for epoch in range(100):
    if len(S_r_x) == 0:
        print("No misclassified samples found in training set")
        break
    fine_optimizer.zero_grad()
    outputs = wisdom_net(S_r_x)
    loss = criterion(outputs, S_r_y)
    loss.backward()
    fine_optimizer.step()

wisdom_net.eval()
with torch.no_grad():
    test_outputs = wisdom_net(X_test_t)
    test_preds = torch.argmax(test_outputs, dim=1)

total_samples = len(y_test_t)
rejected_mask = (test_preds == REJECT_LABEL)
num_rejected = rejected_mask.sum().item()

decided_mask = ~rejected_mask
decided_preds = test_preds[decided_mask]
decided_targets = y_test_t[decided_mask]

misclassified = (decided_preds != decided_targets).sum().item()

error_rate = (misclassified / total_samples) * 100
reject_rate = (num_rejected / total_samples) * 100
decided_accuracy = ((decided_preds == decided_targets).sum().item() / len(decided_targets) * 100) if len(decided_targets) > 0 else 0

print("Final Evaluation Results:")
print(f"Total Test Samples: {total_samples}")
print(f"Rejected Samples: {num_rejected} ({reject_rate:.2f}%)")
print(f"Misclassified Decided Samples: {misclassified}")
print(f"WisdomNet Effective Error Rate: {error_rate:.2f}%")
print(f"Accuracy on Decided Samples: {decided_accuracy:.2f}%")

Pre-training Base Network
BaseNet Test Accuracy: 95.00%, Test Error: 5.00%

Identified 26 misclassified training samples for fine-training (Reject Set Sr).

 Fine-training WisdomNet on Reject Set:
Final Evaluation Results:
Total Test Samples: 240
Rejected Samples: 240 (100.00%)
Misclassified Decided Samples: 0
WisdomNet Effective Error Rate: 0.00%
Accuracy on Decided Samples: 0.00%


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np


torch.manual_seed(42)
np.random.seed(42)

N_SAMPLES = 2000

TEST_SIZE = 1000

INPUT_DIM = 2
HIDDEN_DIM = 8
NUM_CLASSES = 2

REJECT_LABEL = 2

LEARNING_RATE = 0.005

MAX_FINE_EPOCHS = 100

class BaseNet(nn.Module):

    def __init__(
        self,
        input_dim=2,
        hidden_dim=8,
        num_classes=2
    ):
        super(BaseNet, self).__init__()

        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        self.out_layer = nn.Linear(
            hidden_dim,
            num_classes
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        output = self.out_layer(features)
        return output


class WisdomNet(nn.Module):
    def __init__(self, base_model):
        super(WisdomNet, self).__init__()
        self.feature_extractor = base_model.feature_extractor
        hidden_dim = base_model.out_layer.in_features
        self.out_layer = nn.Linear(
            hidden_dim,
            3
        )

        with torch.no_grad():

            self.out_layer.weight[:2, :] = (
                base_model.out_layer.weight.detach()
            )

            self.out_layer.bias[:2] = (
                base_model.out_layer.bias.detach()
            )

            self.out_layer.weight[2, :] = 0.0
            self.out_layer.bias[2] = 0.0

    def forward(self, x):
        features = self.feature_extractor(x)
        output = self.out_layer(features)

        return output


def create_moon_dataset(noise):

    print(f"Creating MOON dataset with noise = {noise}")

    X, y = make_moons(
        n_samples=N_SAMPLES,
        noise=noise,
        random_state=42
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=42,
        stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    X_train_t = torch.tensor(
        X_train,
        dtype=torch.float32
    )

    y_train_t = torch.tensor(
        y_train,
        dtype=torch.long
    )

    X_test_t = torch.tensor(
        X_test,
        dtype=torch.float32
    )

    y_test_t = torch.tensor(
        y_test,
        dtype=torch.long
    )

    print(f"Training samples: {len(X_train_t)}")
    print(f"Testing samples : {len(X_test_t)}")

    return (X_train_t,y_train_t,X_test_t,y_test_t)


def train_base_network(
    X_train_t,
    y_train_t,
    X_test_t,
    y_test_t
):

    base_net = BaseNet(
        input_dim=INPUT_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=NUM_CLASSES
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        base_net.parameters(),
        lr=0.01
    )

    print("Pre-training BaseNet")
    for epoch in range(1000):
        base_net.train()
        optimizer.zero_grad()
        outputs = base_net(X_train_t)
        loss = criterion(
            outputs,
            y_train_t
        )

        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            base_net.eval()
            with torch.no_grad():
                train_outputs = base_net(
                    X_train_t
                )
                train_preds = torch.argmax(
                    train_outputs,
                    dim=1
                )
                train_accuracy = (
                    train_preds == y_train_t
                ).float().mean().item()
                test_outputs = base_net(
                    X_test_t
                )
                test_preds = torch.argmax(
                    test_outputs,
                    dim=1
                )
                test_accuracy = (
                    test_preds == y_test_t
                ).float().mean().item()
            if test_accuracy >= 0.98:

                print(f"BaseNet stopped at epoch {epoch}")

                break

    base_net.eval()
    with torch.no_grad():
        train_outputs = base_net(
            X_train_t
        )
        train_preds = torch.argmax(
            train_outputs,
            dim=1
        )

        train_accuracy = (
            train_preds == y_train_t
        ).float().mean().item()

        test_outputs = base_net(
            X_test_t
        )

        test_preds = torch.argmax(
            test_outputs,
            dim=1
        )

        test_accuracy = (
            test_preds == y_test_t
        ).float().mean().item()

    print(f"BaseNet Training Accuracy : " f"{train_accuracy * 100:.2f}%")

    print(f"BaseNet Testing Accuracy  : " f"{test_accuracy * 100:.2f}%")

    print(f"BaseNet Testing Error     : " f"{(1 - test_accuracy) * 100:.2f}%")

    return base_net


def create_reject_set(
    base_net,
    X_train_t,
    y_train_t
):

    base_net.eval()

    with torch.no_grad():

        train_outputs = base_net(
            X_train_t
        )

        train_preds = torch.argmax(
            train_outputs,
            dim=1
        )

        misclassified_mask = (
            train_preds != y_train_t
        )

    S_r_x = X_train_t[
        misclassified_mask
    ]

    S_r_y = torch.full(
        (S_r_x.size(0),),
        REJECT_LABEL,
        dtype=torch.long
    )

    print("\n Reject Set")

    print(
        f"Training samples             : "
        f"{len(X_train_t)}"
    )

    print(
        f"BaseNet misclassified samples: "
        f"{len(S_r_x)}"
    )

    reject_percentage = (
        len(S_r_x) / len(X_train_t)
    ) * 100

    print(
        f"Reject-set percentage        : "
        f"{reject_percentage:.2f}%"
    )

    return S_r_x, S_r_y


def evaluate_wisdomnet(
    wisdom_net,
    X_test_t,
    y_test_t
):

    wisdom_net.eval()

    with torch.no_grad():

        outputs = wisdom_net(
            X_test_t
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )


    rejected_mask = (
        predictions == REJECT_LABEL
    )

    num_rejected = (
        rejected_mask.sum().item()
    )

    total_samples = len(y_test_t)

    reject_rate = (
        num_rejected / total_samples
    )

    decided_mask = ~rejected_mask
    decided_predictions = predictions[
        decided_mask
    ]

    decided_targets = y_test_t[
        decided_mask
    ]

    num_decided = len(
        decided_targets
    )


    if num_decided > 0:

        num_misclassified = (
            decided_predictions != decided_targets
        ).sum().item()

        decided_accuracy = (
            (decided_predictions == decided_targets)
            .float()
            .mean()
            .item()
        )

    else:

        num_misclassified = 0
        decided_accuracy = 0.0

    effective_error = (
        num_misclassified /
        total_samples
    )


    coverage = (
        num_decided /
        total_samples
    )

    return {
        "total": total_samples,
        "rejected": num_rejected,
        "reject_rate": reject_rate,
        "decided": num_decided,
        "misclassified": num_misclassified,
        "error_rate": effective_error,
        "decided_accuracy": decided_accuracy,
        "coverage": coverage
    }


def fine_train_wisdomnet(
    wisdom_net,
    S_r_x,
    S_r_y,
    X_test_t,
    y_test_t
):

    if len(S_r_x) == 0:

        print("\nNo misclassified training samples.")

        return wisdom_net

    criterion = nn.CrossEntropyLoss()

    fine_optimizer = optim.Adam(
        wisdom_net.parameters(),
        lr=LEARNING_RATE
    )

    print("\n Fine-training WisdomNet")

    history = []
    for epoch in range(
        MAX_FINE_EPOCHS
    ):

        wisdom_net.train()

        fine_optimizer.zero_grad()

        outputs = wisdom_net(
            S_r_x
        )

        loss = criterion(
            outputs,
            S_r_y
        )
        loss.backward()
        fine_optimizer.step()

        results = evaluate_wisdomnet(
            wisdom_net,
            X_test_t,
            y_test_t
        )

        results["epoch"] = epoch + 1
        results["loss"] = loss.item()

        history.append(results)
        if (
            epoch == 0
            or (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch {epoch + 1:3d} | "
                f"Loss {loss.item():.4f} | "
                f"Reject "
                f"{results['reject_rate'] * 100:6.2f}% | "
                f"Error "
                f"{results['error_rate'] * 100:6.2f}% | "
                f"Coverage "
                f"{results['coverage'] * 100:6.2f}%"
            )

        if (
            results["error_rate"] == 0
            and
            results["reject_rate"] <= 0.10
        ):

            print("\n Suitable stopping point found:")

            print(f"Epoch: {epoch + 1}")

            print(f"Error: " f"{results['error_rate'] * 100:.2f}%")

            print(f"Reject rate: " f"{results['reject_rate'] * 100:.2f}%" )

            break

        if results["reject_rate"] >= 0.99:

            print("\n Warning: WisdomNet is approaching all-reject behavior:" )

            print("Stopping fine-training.")

            break

    return wisdom_net, history


def run_experiment(noise):

    print(f"\n\n WISDOMNET MOON EXPERIMENT | " f"Gaussian noise = {noise}" )

    (
        X_train_t,
        y_train_t,
        X_test_t,
        y_test_t
    ) = create_moon_dataset(
        noise
    )


    base_net = train_base_network(
        X_train_t,
        y_train_t,
        X_test_t,
        y_test_t
    )


    (
        S_r_x,
        S_r_y
    ) = create_reject_set(
        base_net,
        X_train_t,
        y_train_t
    )

    wisdom_net = WisdomNet(
        base_net
    )


    wisdom_net, history = fine_train_wisdomnet(
        wisdom_net,
        S_r_x,
        S_r_y,
        X_test_t,
        y_test_t
    )


    final_results = evaluate_wisdomnet(
        wisdom_net,
        X_test_t,
        y_test_t
    )

    print("\n FINAL RESULTS::")

    print(f"Noise: {noise}" )

    print(f"Total Test Samples: " f"{final_results['total']}" )

    print(f"Rejected Samples:" f"{final_results['rejected']}")

    print(f"Reject Rate:" f"{final_results['reject_rate'] * 100:.2f}%")

    print(f"Decided Samples:" f"{final_results['decided']}")

    print(
        f"Misclassified Decided  : "
        f"{final_results['misclassified']}"
    )

    print(
        f"Effective Error Rate   : "
        f"{final_results['error_rate'] * 100:.2f}%"
    )

    print(
        f"Accuracy on Decided    : "
        f"{final_results['decided_accuracy'] * 100:.2f}%"
    )

    print(
        f"Coverage               : "
        f"{final_results['coverage'] * 100:.2f}%"
    )

    return {
        "base_net": base_net,
        "wisdom_net": wisdom_net,
        "history": history,
        "final_results": final_results
    }

results_005 = run_experiment(
    noise=0.05
)

results_010 = run_experiment(
    noise=0.10
)

print("\n\n PAPER-STYLE SUMMARY:::")

for noise, result in [
    (0.05, results_005),
    (0.10, results_010)
]:

    r = result["final_results"]

    print(f"\nMOON noise = {noise}")

    print(f"Reject rate : " f"{r['reject_rate'] * 100:.2f}%")

    print(f"Error rate:"f"{r['error_rate'] * 100:.2f}%")

    print(f"Coverage: "f"{r['coverage'] * 100:.2f}%")

    print(f"Decided acc:" f"{r['decided_accuracy'] * 100:.2f}%")



 WISDOMNET MOON EXPERIMENT | Gaussian noise = 0.05
Creating MOON dataset with noise = 0.05
Training samples: 1000
Testing samples : 1000
Pre-training BaseNet
BaseNet stopped at epoch 80
BaseNet Training Accuracy : 98.90%
BaseNet Testing Accuracy  : 99.10%
BaseNet Testing Error     : 0.90%

 Reject Set
Training samples             : 1000
BaseNet misclassified samples: 11
Reject-set percentage        : 1.10%

 Fine-training WisdomNet
Epoch   1 | Loss 0.8039 | Reject   6.70% | Error   0.00% | Coverage  93.30%

 Suitable stopping point found:
Epoch: 1
Error: 0.00%
Reject rate: 6.70%

 FINAL RESULTS::
Noise: 0.05
Total Test Samples: 1000
Rejected Samples:67
Reject Rate:6.70%
Decided Samples:933
Misclassified Decided  : 0
Effective Error Rate   : 0.00%
Accuracy on Decided    : 100.00%
Coverage               : 93.30%


 WISDOMNET MOON EXPERIMENT | Gaussian noise = 0.1
Creating MOON dataset with noise = 0.1
Training samples: 1000
Testing samples : 1000
Pre-training BaseNet
BaseNet stopped at